# Data Pipeline Verification Sandbox

This notebook now performs two roles:

1. Build the processed DEMADICS `.npy` dataset from the untouched raw DAMADICS archives.
2. Verify every active FlowMatch-PdM data pipeline with one real training batch.

DEMADICS sources used for the preprocessing contract:
- Local description manual: `datasets/Damadics/damadics-lublin-data-description-v02March2002.pdf`
- Local benchmark document: `datasets/Damadics/damadics-benchmark-definition.zip` -> `damadics-benchmark-definition-v1_0.pdf`
- Research context: Step III real-process benchmark papers describe 25 daily files, 4 faulty days, and 19 labeled artificial faults grouped into `f16`, `f17`, `f18`, and `f19`.

FlowMatch-PdM DEMADICS contract used here:
- task: 5-class classification (`normal`, `f16`, `f17`, `f18`, `f19`)
- input: all 32 non-timestamp process variables
- window size: 2048
- split: strict stratified 80/10/10
- scaling: z-score fit on train split only
- fault windows: event-centered sampling from the labeled fault intervals, capped per event so the long open-ended `f17` case does not dominate the dataset


In [1]:
from pathlib import Path
import sys

import torch
import yaml

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "configs" / "default_config.yaml").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils.data_helper import get_data_module, get_dataset_config
from src.utils.demadics_preprocessing import (
    DEMADICS_CLASS_TO_INDEX,
    DEMADICS_FAULT_EVENTS,
    demadics_paths,
    prepare_demadics_processed,
)

CONFIG = yaml.safe_load((REPO_ROOT / "configs" / "default_config.yaml").read_text())
BATCH_SIZE = 8

print(f"Repo root: {REPO_ROOT}")
print(f"Torch: {torch.__version__}")


/home/buddhiw/miniconda3/envs/flowmatch_pdm/lib/python3.10/site-packages/lightning_fabric/__init__.py:41: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


/home/buddhiw/miniconda3/envs/flowmatch_pdm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repo root: /home/buddhiw/flowmatch/FlowMatch-PdM
Torch: 2.2.1+cu121


In [2]:
FORCE_REBUILD_DEMADICS = False

paths = demadics_paths(REPO_ROOT)
metadata = prepare_demadics_processed(REPO_ROOT, force_rebuild=FORCE_REBUILD_DEMADICS)

print("DEMADICS preprocessing ready")
print(f"Archive dir:   {paths['archive_dir']}")
print(f"Raw dir:       {metadata['raw_dir']}")
print(f"Processed dir: {metadata['processed_dir']}")
print(f"Fault events encoded from manual: {len(DEMADICS_FAULT_EVENTS)}")
print(f"Class map: {DEMADICS_CLASS_TO_INDEX}")
print(f"Split sizes -> train: {metadata['train_size']}, val: {metadata['val_size']}, test: {metadata['test_size']}")
print(f"Train class counts: {metadata['train_class_counts']}")
print(f"Val class counts:   {metadata['val_class_counts']}")
print(f"Test class counts:  {metadata['test_class_counts']}")


DEMADICS preprocessing ready
Archive dir:   /home/buddhiw/flowmatch/datasets/Damadics
Raw dir:       /home/buddhiw/flowmatch/datasets/damadics_raw/Lublin_all_data
Processed dir: /home/buddhiw/flowmatch/datasets/processed/demadics
Fault events encoded from manual: 19
Class map: {'normal': 0, 'f16': 1, 'f17': 2, 'f18': 3, 'f19': 4}
Split sizes -> train: 339, val: 42, test: 43
Train class counts: {'normal': 169, 'f16': 57, 'f17': 30, 'f18': 57, 'f19': 26}
Val class counts:   {'normal': 21, 'f16': 7, 'f17': 4, 'f18': 7, 'f19': 3}
Test class counts:  {'normal': 22, 'f16': 7, 'f17': 3, 'f18': 8, 'f19': 3}


In [3]:
SUPPORTED_DATASETS = [
    {"dataset": "CMAPSS", "track": "engine_rul", "fd": 1},
    {"dataset": "N-CMAPSS", "track": "engine_rul", "fd": 1},
    {"dataset": "FEMTO", "track": "bearing_rul", "fd": 1},
    {"dataset": "XJTU-SY", "track": "bearing_rul", "fd": 1},
    {"dataset": "CWRU", "track": "bearing_fault"},
    {"dataset": "Paderborn", "track": "bearing_fault"},
    {"dataset": "DEMADICS", "track": "bearing_fault"},
]

def build_datamodule(spec: dict, batch_size: int = BATCH_SIZE):
    dataset_cfg = get_dataset_config(CONFIG, spec["dataset"])
    kwargs = {
        "track": spec["track"],
        "dataset_name": spec["dataset"],
        "window_size": dataset_cfg["window_size"],
        "batch_size": batch_size,
    }
    if "fd" in spec:
        kwargs["fd"] = spec["fd"]
    return get_data_module(**kwargs)

def assert_batch_window_first(x_batch: torch.Tensor, expected_window: int, dataset_name: str) -> None:
    assert x_batch.ndim == 3, f"{dataset_name}: expected a 3D batch, got shape {tuple(x_batch.shape)}"
    assert x_batch.shape[1] == expected_window, f"{dataset_name}: expected [batch, {expected_window}, features], got {tuple(x_batch.shape)}"
    assert x_batch.shape[2] > 0, f"{dataset_name}: feature dimension must be positive"

def assert_target_dtype(y_batch: torch.Tensor, track: str, dataset_name: str) -> None:
    if "rul" in track:
        assert torch.is_floating_point(y_batch), f"{dataset_name}: expected floating-point RUL targets, got {y_batch.dtype}"
        return
    integer_dtypes = {torch.int8, torch.int16, torch.int32, torch.int64, torch.uint8}
    assert y_batch.dtype in integer_dtypes, f"{dataset_name}: expected integer classification targets, got {y_batch.dtype}"


In [4]:
results = []

for spec in SUPPORTED_DATASETS:
    dataset_name = spec["dataset"]
    dataset_cfg = get_dataset_config(CONFIG, dataset_name)
    print("=" * 96)
    print(f"Dataset Name: {dataset_name}")
    try:
        dm = build_datamodule(spec)
        dm.prepare_data()
        dm.setup("fit")
        x_batch, y_batch = next(iter(dm.train_dataloader()))
        assert_batch_window_first(x_batch, dataset_cfg["window_size"], dataset_name)
        assert_target_dtype(y_batch, spec["track"], dataset_name)
        minority_ds = dm.get_minority_dataset()
        print(f"X_train_batch.shape: {tuple(x_batch.shape)}")
        print(f"y_train_batch.shape: {tuple(y_batch.shape)}")
        print(f"y_train_batch.dtype: {y_batch.dtype}")
        print(f"Minority subset size: {len(minority_ds)} / {len(dm.train_ds)}")
        results.append({"dataset": dataset_name, "status": "PASS", "message": ""})
    except Exception as exc:
        message = f"{type(exc).__name__}: {exc}"
        print(f"Verification failed: {message}")
        results.append({"dataset": dataset_name, "status": "FAIL", "message": message})

print("\n" + "=" * 96)
print("Verification Summary")
for result in results:
    suffix = f" ({result['message']})" if result["message"] else ""
    print(f"- {result['dataset']}: {result['status']}{suffix}")
supported_failures = [result for result in results if result["status"] != "PASS"]
print(f"\nSupported loader readiness: {'GO' if not supported_failures else 'NO-GO'}")
print(f"Full requested roster readiness: {'GO' if not supported_failures else 'NO-GO'}")


Dataset Name: CMAPSS


[CMAPSS] Extracted 2000 minority samples (RUL <= 25.0000) across conditions [1]
X_train_batch.shape: (8, 30, 14)
y_train_batch.shape: (8,)
y_train_batch.dtype: torch.float32
Minority subset size: 2000 / 13818
Dataset Name: N-CMAPSS


[N-CMAPSS] Extracted 70 minority samples (RUL <= 13.0000) across conditions [1]
X_train_batch.shape: (8, 50, 32)
y_train_batch.shape: (8,)
y_train_batch.dtype: torch.float32
Minority subset size: 70 / 459
Dataset Name: FEMTO


[FEMTO] Extracted 1120 minority samples (RUL <= 560.6000) across conditions [1]
X_train_batch.shape: (8, 2560, 2)
y_train_batch.shape: (8,)
y_train_batch.dtype: torch.float32
Minority subset size: 1120 / 3674
Dataset Name: XJTU-SY


[XJTU-SY] Extracted 64 minority samples (RUL <= 32.2000) across conditions [1]
X_train_batch.shape: (8, 2048, 2)
y_train_batch.shape: (8,)
y_train_batch.dtype: torch.float32
Minority subset size: 64 / 284
Dataset Name: CWRU
[CWRU] Loaded from /home/buddhiw/flowmatch/datasets/processed/cwru/  Train: 7573 | Val: 947 | Test: 947


[CWRU] Minority label(s): [0, 8] with 754 samples each. Extracted 1508 samples.
X_train_batch.shape: (8, 2048, 1)
y_train_batch.shape: (8,)
y_train_batch.dtype: torch.int64
Minority subset size: 1508 / 7573
Dataset Name: Paderborn


[Paderborn] Loaded from /home/buddhiw/flowmatch/datasets/processed/paderborn/  Train: 491325 | Val: 61416 | Test: 61416


[Paderborn] Minority label(s): [20] with 15816 samples each. Extracted 15816 samples.
X_train_batch.shape: (8, 4096, 1)
y_train_batch.shape: (8,)
y_train_batch.dtype: torch.int64
Minority subset size: 15816 / 491325
Dataset Name: DEMADICS


[DEMADICS] Loaded from /home/buddhiw/flowmatch/datasets/processed/demadics/  Train: 339 | Val: 42 | Test: 43


[DEMADICS] Minority label(s): [4] with 26 samples each. Extracted 26 samples.
X_train_batch.shape: (8, 2048, 32)
y_train_batch.shape: (8,)
y_train_batch.dtype: torch.int64
Minority subset size: 26 / 339

Verification Summary
- CMAPSS: PASS
- N-CMAPSS: PASS
- FEMTO: PASS
- XJTU-SY: PASS
- CWRU: PASS
- Paderborn: PASS
- DEMADICS: PASS

Supported loader readiness: GO
Full requested roster readiness: GO
